In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import numpy as np
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(
    "lighteternal/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext-finetuned-mnli"
)
model = AutoModelForSequenceClassification.from_pretrained(
    "lighteternal/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext-finetuned-mnli"
).to(device)

LABEL_MAP = {0: "entailment", 1: "neutral", 2: "contradiction"}

def classify_nli(premise, hypothesis):
    x = tokenizer.encode(premise, hypothesis, return_tensors='pt', truncation=True).to(device)
    logits = model(x)[0]
    probs = logits.softmax(dim=1).cpu().detach().numpy()[0]
    result = {LABEL_MAP[i]: round(float(p), 3) for i, p in enumerate(probs)}
    return {"probs": result, "predicted": max(result, key=result.get)}

test_cases = [
    # ENTAILMENT
    ("The patient was diagnosed with type 2 diabetes mellitus.",
     "The patient has a metabolic disorder.", "entailment", "Semantic entailment"),
    ("Patient presents with chest pain radiating to the left arm and diaphoresis.",
     "Patient may be experiencing a myocardial infarction.", "entailment", "Symptom inference"),
    ("The biopsy confirmed malignant cells in the lymph nodes.",
     "The patient has cancer that has spread to the lymphatic system.", "entailment", "Oncology entailment"),
    ("Patient is a current smoker with 40 pack-year history.",
     "Patient has significant smoking history.", "entailment", "Risk factor entailment"),
    ("The patient is on metformin and insulin therapy.",
     "The patient is receiving pharmacological treatment for diabetes.", "entailment", "Treatment entailment"),
    ("Patient has a heart rate of 115bpm.",
     "Patient is tachycardic.", "entailment", "Numerical raw"),
    ("Patient BMI is 32.",
     "Patient is obese.", "entailment", "Numerical raw"),
    ("SpO2 is 88%.",
     "Patient is hypoxic.", "entailment", "Numerical raw"),
    ("Patient sodium is 128mEq/L.",
     "Patient has hyponatremia.", "entailment", "Numerical raw"),
    ("WBC count is 14,500 cells/mcL.",
     "Patient has leukocytosis.", "entailment", "Numerical raw"),
    # CONTRADICTION
    ("Patient has no known drug allergies.",
     "Patient is allergic to penicillin.", "contradiction", "Allergy contradiction"),
    ("The patient's renal function is within normal limits.",
     "The patient has chronic kidney disease.", "contradiction", "Renal contradiction"),
    ("Patient is a lifelong non-smoker.",
     "Patient has a history of tobacco use.", "contradiction", "Smoking contradiction"),
    ("The tumor was completely resected with clear margins.",
     "Residual tumor remains after surgery.", "contradiction", "Surgical contradiction"),
    ("Patient has been in remission for 5 years.",
     "Patient currently has active disease.", "contradiction", "Remission contradiction"),
    # NEUTRAL
    ("Patient is prescribed lisinopril 10mg daily.",
     "Patient has a history of falls.", "neutral", "Unrelated neutral"),
    ("The patient underwent appendectomy in 2018.",
     "The patient is currently taking statins.", "neutral", "Unrelated neutral"),
    ("Patient reports fatigue and weight gain.",
     "Patient lives alone.", "neutral", "Social neutral"),
    ("MRI showed a herniated disc at L4-L5.",
     "Patient has a family history of heart disease.", "neutral", "Unrelated neutral"),
    ("Patient is post-operative day 2 following knee replacement.",
     "Patient has a documented latex allergy.", "neutral", "Unrelated neutral"),
]

results = []
correct = 0

for i, (premise, hypothesis, expected, category) in enumerate(test_cases):
    out = classify_nli(premise, hypothesis)
    match = out["predicted"] == expected
    if match:
        correct += 1
    results.append({
        "id": i + 1, "category": category,
        "premise": premise, "hypothesis": hypothesis,
        "expected": expected, "predicted": out["predicted"],
        "probs": out["probs"], "correct": match,
    })

print(f"\n{'='*70}")
print(f"RESULTS: {correct}/20 correct ({correct * 5}% accuracy)")
print(f"{'='*70}\n")

for r in results:
    status = "✅" if r["correct"] else "❌"
    print(f"[{r['id']:02d}] {status} {r['category']}")
    print(f"     P: {r['premise']}")
    print(f"     H: {r['hypothesis']}")
    print(f"     Expected: {r['expected']:15s} | Predicted: {r['predicted']}")
    e, n, c = r['probs']['entailment'], r['probs']['neutral'], r['probs']['contradiction']
    print(f"     Ent={e:.3f}  Neu={n:.3f}  Con={c:.3f}")
    print()

print(f"{'='*70}")
print("ACCURACY BY LABEL:")
by_label = defaultdict(list)
for r in results:
    by_label[r['expected']].append(r['correct'])
for label, vals in by_label.items():
    print(f"  {label:15s}: {sum(vals)}/{len(vals)}  ({sum(vals)/len(vals)*100:.0f}%)")